# 🚀 MMLU 評測工具箱 (Google Colab GPU / 本地通用版)
### Pegatron LLM Benchmark Suite — 4 大 Adapter 與 Log-Likelihood 標準評測

本 Notebook 專為 **Google Colab (免費 T4 GPU / A100 / L4) 或本地工作站** 打造，全面採用 **學術標準 Next-Token Log-Likelihood (條件對數機率)** 進行選擇題評測，支援以下 **4 種工程師評測 Adapter**：

1. 🧩 **Hugging Face Adapter (`huggingface`)**：直接載入 Hugging Face Hub 或本地訓練好的 Checkpoint (支援 4-bit NF4 量化)。
2. 🦙 **llama.cpp / GGUF Adapter (`llamacpp`)**：載入 `.gguf` 邊緣量化模型，支援 GPU Layer Offload 與 Logprob 評測。
3. ⚙️ **自訂 Python 函式 Adapter (`custom`)**：允許工程師接入任何自研 PyTorch Module、ONNX、或特化 Scoring Pipeline。
4. 🧪 **Mock 基準 Adapter (`mock`)**：0 耗時、0 GPU、0 網路快速驗證管線與報表格式。

---

## 步驟 1: 環境檢查與套件安裝

In [ ]:
# 1. 檢查 GPU 狀態 (建議在 Colab 選單選擇「執行階段」->「變更執行階段類型」->「T4 GPU」)
!nvidia-smi

# 2. 安裝 Hugging Face、Transformers、4-bit 量化與報表繪圖套件
!pip install -q torch transformers datasets accelerate bitsandbytes matplotlib numpy pandas tabulate

# 3. 若欲測試 Option 2 (GGUF / llama.cpp)，可直接安裝預編譯 CUDA 12.2 GPU 加速版：
# !pip install --no-cache-dir llama-cpp-python==0.3.16 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122


## 步驟 2: 下載或匯入 MMLU Benchmark 工具庫

In [ ]:
import os, sys

# 確保當前目錄在 Python 模組搜尋路徑中
if '.' not in sys.path:
    sys.path.insert(0, '.')

from mmlu_benchmark.benchmark import BenchmarkConfig, MMLUBenchmarkPipeline
from mmlu_benchmark.dataset import BALANCED_BENCHMARK_SUBJECTS, ALL_57_SUBJECTS
from mmlu_benchmark.report_generator import ReportGenerator

print("✅ MMLU Benchmark Suite 模組載入成功！")

--- 
## 🛠️ 四大 Adapter 評測範例 (工程師可任選一種或多種執行)

### 🔹 選項 1: 使用 Hugging Face Adapter 評測 (Transformers / 4-bit NF4)
> 適用於評測開源模型 (例如 `Qwen/Qwen2.5-0.5B-Instruct`、`meta-llama/Meta-Llama-3-8B-Instruct`) 或自己的 LoRA / SFT Checkpoint。

In [ ]:
import asyncio

async def run_huggingface_eval():
    config = BenchmarkConfig(
        model_name="Qwen/Qwen2.5-0.5B-Instruct",  # 可替換為任意 HF 模型或本地 checkpoint 路徑
        provider="huggingface",
        subjects=BALANCED_BENCHMARK_SUBJECTS,    # 四大領域平衡 12 學科
        shots=5,                                 # 5-Shot 提示
        load_in_4bit=True,                       # 啟用 4-bit 量化降低顯存需求
        preload_dataset=True,                    # 單批次預載資料集，杜絕頻率限制與卡頓
        report_dir="./report_hf",
    )
    pipeline = MMLUBenchmarkPipeline(config=config)
    metrics = await pipeline.run_benchmark()
    print("\n" + metrics.to_markdown_summary())
    return metrics

# 執行評測
hf_metrics = await run_huggingface_eval()

### 🔹 選項 2: 使用 llama.cpp / GGUF Adapter 評測 (`.gguf` 檔案)
> 適用於在邊緣端或本機直接評測量化好的 GGUF 模型 (例如 `./Qwen2.5-1.5B-Instruct-Q4_K_M.gguf`)。

In [ ]:
# 💡 [工程師測試碼] 1. 安裝預編譯 CUDA 12.2 GPU 加速版 llama-cpp-python
!pip install --no-cache-dir llama-cpp-python==0.3.16 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

# 💡 [工程師測試碼] 2. 下載官方 Qwen2.5-1.5B-Instruct GGUF 量化模型 (Q4_K_M，約 1.1GB)
!wget -O Qwen2.5-1.5B-Instruct-Q4_K_M.gguf https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf

In [ ]:
async def run_llamacpp_eval():
    # 請確保已下載 .gguf 檔案至當前目錄 (可執行上方儲存格一鍵下載)
    gguf_path = "./Qwen2.5-1.5B-Instruct-Q4_K_M.gguf"
    
    if not os.path.exists(gguf_path):
        print(f"⚠️ 找不到 {gguf_path}，請先執行上方測試碼下載 GGUF 檔案。此處先以 Mock 回退示範。")
        config = BenchmarkConfig(model_name="mock-gguf", provider="mock", preset="balanced", max_samples_per_subject=5)
    else:
        config = BenchmarkConfig(
            model_name=gguf_path,
            provider="llamacpp",
            preset="balanced",
            batch_size=1,  # 嚴格設定 batch_size=1 確保 C++ KV-cache 原生線程安全
            shots=5,
            preload_dataset=True,
            report_dir="./report_gguf",
        )
    
    pipeline = MMLUBenchmarkPipeline(config=config)
    metrics = await pipeline.run_benchmark()
    print("\n" + metrics.to_markdown_summary())
    return metrics

# 執行評測
gguf_metrics = await run_llamacpp_eval()

### 🔹 選項 3: 使用自訂 Python 函式 Adapter (`custom`)
> 允許工程師自訂任意推論函式（回傳各選項 Log-Likelihood 對數機率字典 `{"A": -0.5, "B": -1.2, ...}`，由評測管線執行學術標準 ArgMax 判定）。

In [ ]:
# 定義工程師自訂的評測 / 推論邏輯
def my_custom_scoring_fn(prompt: str) -> dict:
    """
    工程師可在此處執行 PyTorch Forward、ONNX Runtime、vLLM API 端點、或自研神經網絡。
    必須回傳學術標準條件對數機率 (Log-Likelihood) 或 Logits 字典：
      {"A": -1.5, "B": -0.2, "C": -2.8, "D": -3.1}
    由評測管線自動執行確定性 ArgMax 選項判定（嚴格對齊 NeurIPS 論文標準，杜絕正則字串解析誤差）。
    """
    # 示範回傳 Log-Likelihood 分數字典 (A, B, C, D)
    return {"A": -2.1, "B": -0.15, "C": -3.4, "D": -4.2}

async def run_custom_eval():
    config = BenchmarkConfig(
        model_name="in-house-custom-model",
        provider="custom",
        preset="balanced",  # 支援 "balanced" (12科), "full" (57科), 或 "quick" (2科)
        shots=5,
        max_samples_per_subject=5,
        report_dir="./report_custom",
    )
    pipeline = MMLUBenchmarkPipeline(config=config, custom_fn=my_custom_scoring_fn)
    metrics = await pipeline.run_benchmark()
    print("\n" + metrics.to_markdown_summary())
    return metrics

# 執行評測
custom_metrics = await run_custom_eval()

### 🔹 選項 4: 使用 Mock Adapter 快速驗證 (`mock`)
> 0 依賴、0 顯存需求，秒級完成驗證。

In [ ]:
async def run_mock_eval():
    config = BenchmarkConfig(
        model_name="mock-baseline",
        provider="mock",
        subjects=BALANCED_BENCHMARK_SUBJECTS,
        shots=5,
        max_samples_per_subject=3,
        report_dir="./report_mock",
    )
    pipeline = MMLUBenchmarkPipeline(config=config)
    metrics = await pipeline.run_benchmark()
    print("\n" + metrics.to_markdown_summary())
    return metrics

# 執行評測
mock_metrics = await run_mock_eval()

--- 
## 步驟 3: 產生跨模型四大領域雷達圖與長條圖視覺化報表

In [ ]:
from colab_run_full_benchmark import plot_comparison_figures

# 彙整所有已評測模型的結果
all_evaluated_results = {
    "Qwen2.5-0.5B": hf_metrics.to_dict(),
    "Custom-Model": custom_metrics.to_dict(),
    "Mock-Baseline": mock_metrics.to_dict(),
}

# 輸出 4 種 300 DPI 圖表
plot_comparison_figures(all_evaluated_results, "./report_charts")
print("🎉 評測完成！圖表與 HTML 報告已儲存至 ./report_charts")